# 🎨 Pix2Pix — Image-to-Image Translation
**Final Year Project** · U-Net Generator + PatchGAN Discriminator

This notebook trains on Google Colab free GPU (Tesla T4).

---
**Steps:**
1. Install dependencies
2. Download dataset (facades / edges2shoes / day2night)
3. Train model (~2-3 hours on T4 for 150 epochs)
4. Evaluate with SSIM / PSNR
5. Generate sample outputs
6. Download trained generator for web app

In [ ]:
# ── 1. Mount Drive + Install ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm matplotlib seaborn

In [ ]:
# ── 2. Clone / upload project ─────────────────────────────────
# Option A: upload zip
# from google.colab import files
# uploaded = files.upload()  # upload pix2pix_project.zip
# !unzip pix2pix_project.zip -d /content/

# Option B: if pushed to GitHub
# !git clone https://github.com/YOUR_USERNAME/pix2pix-project /content/pix2pix_project

%cd /content/pix2pix_project
import os; os.listdir('.')

In [ ]:
# ── 3. Download Dataset ───────────────────────────────────────
# Choose ONE of the options below:

# OPTION A: Official Pix2Pix datasets (Berkeley)
DATASET = 'facades'   # options: facades, edges2shoes, edges2handbags, maps, cityscapes
!wget -q http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/{DATASET}.tar.gz
!tar -xzf {DATASET}.tar.gz
!mkdir -p dataset/train dataset/val
!mv {DATASET}/train/* dataset/train/
!mv {DATASET}/val/*   dataset/val/
print('Dataset ready!')

# OPTION B: Day2Night (Kaggle)
# !pip install -q kaggle
# from google.colab import files; files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d hayiyo/pix2pix-day-to-night
# !unzip pix2pix-day-to-night.zip

In [ ]:
# ── 4. Check GPU ──────────────────────────────────────────────
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

# Memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# ── 5. Preview dataset samples ────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image

samples = list(Path('dataset/train').glob('*.jpg'))[:4]
fig, axes = plt.subplots(4, 2, figsize=(10, 16))

for i, path in enumerate(samples):
    img = np.array(Image.open(path))
    w   = img.shape[1] // 2
    axes[i, 0].imshow(img[:, :w]);  axes[i, 0].set_title('Input');  axes[i, 0].axis('off')
    axes[i, 1].imshow(img[:, w:]);  axes[i, 1].set_title('Target'); axes[i, 1].axis('off')

plt.suptitle('Dataset Samples', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────
# This will take ~2-3 hours on Colab free T4 for 150 epochs
# Tip: Enable TensorBoard below to monitor live

%load_ext tensorboard
%tensorboard --logdir logs/

!python train.py --epochs 150 --mode paired

In [ ]:
# ── 7. Evaluate — SSIM + PSNR on test set ────────────────────
import tensorflow as tf
import numpy as np
from dataset import make_paired_dataset, tensor_to_image

generator = tf.keras.models.load_model('models/generator.h5', compile=False)
test_ds   = make_paired_dataset('dataset/val', is_train=False)

ssim_scores, psnr_scores = [], []
for inp, tar in test_ds.take(50):
    pred   = generator(inp, training=False)
    p01    = (pred + 1.0) / 2.0
    t01    = (tar  + 1.0) / 2.0
    ssim_scores.append(float(tf.reduce_mean(tf.image.ssim(t01, p01, max_val=1.0))))
    psnr_scores.append(float(tf.reduce_mean(tf.image.psnr(t01, p01, max_val=1.0))))

print(f'SSIM : {np.mean(ssim_scores):.4f}  (1.0 = perfect)')
print(f'PSNR : {np.mean(psnr_scores):.2f} dB  (higher = better)')

In [ ]:
# ── 8. Visual results grid ────────────────────────────────────
import matplotlib.pyplot as plt

n = 6
fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
axes[0, 0].set_title('Input',    fontsize=14)
axes[0, 1].set_title('Generated',fontsize=14)
axes[0, 2].set_title('Target',   fontsize=14)

for i, (inp, tar) in enumerate(test_ds.take(n)):
    pred = generator(inp, training=False)
    axes[i, 0].imshow(tensor_to_image(inp[0]))
    axes[i, 1].imshow(tensor_to_image(pred[0]))
    axes[i, 2].imshow(tensor_to_image(tar[0]))
    for ax in axes[i]: ax.axis('off')

plt.suptitle('Pix2Pix Results — Input | Generated | Target', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('results_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results_grid.png')

In [ ]:
# ── 9. Training curves ────────────────────────────────────────
from IPython.display import Image as IPImage
IPImage('logs/training_curves.png')

In [ ]:
# ── 10. Download trained model ───────────────────────────────
# Save to Google Drive for persistent storage
import shutil
shutil.copy('models/generator.h5', '/content/drive/MyDrive/pix2pix_generator.h5')
print('Saved to Google Drive!')

# OR download directly
# from google.colab import files
# files.download('models/generator.h5')